In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path("..") # Car le notebool est dans le dossier "notebooks"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

GLOBAL_PATH = DATA_RAW / "global"
EUROPE_PATH = DATA_RAW / "europe"
CANADA_PATH = DATA_RAW / "canada"
USA_PATH = DATA_RAW / "usa"
JAPAN_PATH = DATA_RAW / "japan"
UNHCR_PATH = DATA_RAW / "unhcr"
CONTEXT_PATH = DATA_RAW / "context"

In [ ]:
def audit_dataset(
    df,
    dataset_name,
    year_col=None,
    country_cols=None,
    focus_value="Cameroon",
    max_display=10
):
    print("=" * 70)
    print(f"AUDIT : {dataset_name}")
    print("=" * 70)

    # 1. Taille du jeu de données
    print("\n[1] Taille du dataset")
    print(f"Lignes : {df.shape[0]}")
    print(f"Colonnes : {df.shape[1]}")

    # 2. Colonnes
    print("\n[2] Colonnes")
    print(df.columns.tolist())

    # 3. Types
    print("\n[3] Types")
    print(df.dtypes)

    # 4. Valeurs manquantes
    print("\n[4] Valeurs manquantes")
    print(df.isna().sum())

    # 5. Doublons
    print("\n[5] Doublons")
    print(df.duplicated().sum())

    # 6. Aperçu
    print("\n[6] Aperçu")
    display(df.head())

    # 7. Années disponibles
    if year_col and year_col in df.columns:
        print("\n[7] Années disponibles")
        years = sorted(df[year_col].dropna().unique())
        print(years)

    # 8. Pays disponibles
    if country_cols:
        print("\n[8] Pays disponibles")
        for col in country_cols:
            if col in df.columns:
                print(f"\nColonne : {col}")
                countries = df[col].dropna().unique()
                print(f"Nombre de valeurs distinctes : {len(countries)}")
                print(f"Exemples : {list(countries[:max_display])}")

    # 9. Présence de la valeur Cameroon
    if country_cols:
        print("\n[9] Présence de Cameroon")
        for col in country_cols:
            if col in df.columns:
                has_cameroon = df[col].astype(str).str.contains(focus_value, case=False, na=False).any()
                print(f"{col} : {has_cameroon}")

## Audit - UN DESA
Objectif : vérifier la structure du jeu de données principal pour Q1 (destinations mondiales).

In [5]:
undesa = pd.read_csv(GLOBAL_PATH / "undesa_cameroon_global_destinations.csv")
audit_dataset(undesa, "UN DESA")

AUDIT : UN DESA

[1] Taille du dataset
Lignes : 224240
Colonnes : 8

[2] Colonnes
['Destination', 'Location code of destination', 'Origin', 'Location code of origin', 'Year', 'Total', 'Male', 'Female']

[3] Types
Destination                       str
Location code of destination    int64
Origin                            str
Location code of origin         int64
Year                            int64
Total                             str
Male                              str
Female                            str
dtype: object

[4] Valeurs manquantes
Destination                     0
Location code of destination    0
Origin                          0
Location code of origin         0
Year                            0
Total                           1
Male                            0
Female                          0
dtype: int64

[5] Doublons
0

[6] Aperçu


,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
0,World,900,World,900,1990,153 916 063,77 772 082,76 143 981
1,World,900,Sub-Saharan Africa,1834,1990,14 124 662,7 458 958,6 665 704
2,World,900,Northern Africa and Western Asia,1833,1990,14 986 109,8 337 422,6 648 687
3,World,900,Central and Southern Asia,1831,1990,30 342 957,16 862 386,13 480 571
4,World,900,Eastern and South-Eastern Asia,1832,1990,14 465 509,7 069 837,7 395 672


In [6]:
print("Annees disponibles :", sorted(undesa["Year"].unique()))
print("Origines disponibles :", undesa["Origin"].nunique())
print("Destinations disponibles :", undesa["Destination"].nunique())

Annees disponibles : [np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2020), np.int64(2024)]
Origines disponibles : 285
Destinations disponibles : 286


In [7]:
undesa[undesa["Origin"] == "Cameroon"].head()

,Destination,Location code of destination,Origin,Location code of origin,Year,Total,Male,Female
49,World,900,Cameroon,120,1990,92 391,49 277,43 114
335,Sub-Saharan Africa,1834,Cameroon,120,1990,63 197,33 785,29 412
570,Northern Africa and Western Asia,1833,Cameroon,120,1990,65,51,14
765,Central and Southern Asia,1831,Cameroon,120,1990,..,..,..
914,Eastern and South-Eastern Asia,1832,Cameroon,120,1990,315,189,126


### Constats
- Jeu de données principal pour Q1 : il contient le pays d'origine, le pays de destination, l'année et le stock de migrants.
- Structure longue : une ligne représente une observation.
- Années disponibles : 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024. Le jeu de données ne donne pas une série annuelle complète, mais des points de mesure ? certaines années.
- Attention aux agrégats régionaux : en plus des pays, certaines lignes correspondent ? des groupes comme `World`, `Africa`, `Asia`, `Eastern Africa` ou `Western Africa`.
- Convertir les colonnes numériques si besoin.

## Audit - Eurostat premiers permis
Objectif : vérifier les motifs d'entrée en Europe pour les Camerounais.

In [8]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx")
xls.sheet_names

/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Sommaire', 'Feuille 1', 'Feuille 2', 'Feuille 3', 'Feuille 4']

In [13]:
sheet1 = pd.read_excel(EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx", sheet_name="Feuille 1", header = None)
audit_dataset(sheet1, "Eurostat first permits - sheet 1")

AUDIT : Eurostat first permits - sheet 1

[1] Taille du dataset
Lignes : 53
Colonnes : 21

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

[3] Types
0         str
1      object
2         str
3      object
4         str
5      object
6     float64
7      object
8     float64
9      object
10    float64
11     object
12    float64
13     object
14        str
15     object
16        str
17     object
18        str
19     object
20        str
dtype: object

[4] Valeurs manquantes
0      3
1     12
2     47
3     17
4     52
5     17
6     53
7     17
8     53
9     17
10    53
11    17
12    53
13    17
14    52
15    17
16    52
17    17
18    52
19    17
20    52
dtype: int64

[5] Doublons
2

[6] Aperçu


/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,Données extraites le13/04/2026 13:19:36 depuis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,Permis délivrés pour la première fois par rais...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dernière mise à jour:,29/03/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fréquence (relative au temps),NaN,Annuel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Constats
- Plusieurs feuilles sont présentes.
- Une feuille correspond ? chaque type de raison.
- Les vraies données commencent après les lignes de métadonnées.
- Une transformation de la structure large vers une structure longue est nécessaire.
- Présence possible de flags Eurostat.

## Audit - Eurostat résidents de longue durée
Objectif : vérifier le jeu de données des résidents de longue durée pour les Camerounais en Europe.

In [17]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx")
xls.sheet_names

/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3']

In [23]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [22]:
audit_dataset(
    df=sheet1_raw,
    dataset_name="Eurostat long-term residents - Sheet 1 (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : Eurostat long-term residents - Sheet 1 (raw)

[1] Taille du dataset
Lignes : 53
Colonnes : 21

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

[3] Types
0         str
1      object
2         str
3      object
4         str
5      object
6         str
7      object
8     float64
9      object
10    float64
11     object
12    float64
13     object
14        str
15     object
16        str
17     object
18        str
19     object
20        str
dtype: object

[4] Valeurs manquantes
0      3
1     11
2     48
3     17
4     49
5     17
6     52
7     17
8     53
9     17
10    53
11    17
12    53
13    17
14    51
15    17
16    52
17    17
18    52
19    17
20    52
dtype: int64

[5] Doublons
2

[6] Aperçu


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,Data extracted on 09/04/2026 14:04:03 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,Long-term residents by citizenship on 31 Decem...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,23/03/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Constats
- Le fichier contient une feuille `Summary` et 3 feuilles de données.
- Les vraies données ne commencent pas ? la première ligne.
- La feuille 1 correspond au cadre juridique total.
- Les années visibles vont de 2015 ? 2024.
- La nationalit? est déjà filtrée sur `Cameroon`.
- Présence possible de flags Eurostat.

## Audit - Eurostat changements de statut
Objectif : vérifier le jeu de données des changements de statut pour les Camerounais en Europe.

In [24]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_status_changes_cameroon.xlsx")
xls.sheet_names

/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Summary',
 'Sheet 1',
 'Sheet 2',
 'Sheet 3',
 'Sheet 4',
 'Sheet 5',
 'Sheet 6',
 'Sheet 7',
 'Sheet 8',
 'Sheet 9',
 'Sheet 10',
 'Sheet 11',
 'Sheet 12',
 'Sheet 13',
 'Sheet 14',
 'Sheet 15',
 'Sheet 16',
 'Sheet 17',
 'Sheet 18',
 'Sheet 19',
 'Sheet 20',
 'Sheet 21',
 'Sheet 22',
 'Sheet 23',
 'Sheet 24',
 'Sheet 25',
 'Sheet 26',
 'Sheet 27',
 'Sheet 28',
 'Sheet 29',
 'Sheet 30',
 'Sheet 31',
 'Sheet 32',
 'Sheet 33',
 'Sheet 34',
 'Sheet 35',
 'Sheet 36',
 'Sheet 37',
 'Sheet 38',
 'Sheet 39',
 'Sheet 40',
 'Sheet 41',
 'Sheet 42',
 'Sheet 43',
 'Sheet 44',
 'Sheet 45',
 'Sheet 46',
 'Sheet 47',
 'Sheet 48',
 'Sheet 49',
 'Sheet 50',
 'Sheet 51',
 'Sheet 52',
 'Sheet 53',
 'Sheet 54',
 'Sheet 55',
 'Sheet 56',
 'Sheet 57',
 'Sheet 58',
 'Sheet 59',
 'Sheet 60',
 'Sheet 61',
 'Sheet 62',
 'Sheet 63',
 'Sheet 64',
 'Sheet 65',
 'Sheet 66',
 'Sheet 67',
 'Sheet 68',
 'Sheet 69',
 'Sheet 70',
 'Sheet 71',
 'Sheet 72',
 'Sheet 73',
 'Sheet 74',
 'Sheet 75',
 'Sheet 76',
 'Sheet 7

In [25]:
audit_dataset(
    df=sheet1_raw,
    dataset_name="Eurostat status changes - Sheet 1 (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : Eurostat status changes - Sheet 1 (raw)

[1] Taille du dataset
Lignes : 53
Colonnes : 21

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

[3] Types
0         str
1      object
2         str
3      object
4         str
5      object
6         str
7      object
8     float64
9      object
10    float64
11     object
12    float64
13     object
14        str
15     object
16        str
17     object
18        str
19     object
20        str
dtype: object

[4] Valeurs manquantes
0      3
1     11
2     48
3     17
4     49
5     17
6     52
7     17
8     53
9     17
10    53
11    17
12    53
13    17
14    51
15    17
16    52
17    17
18    52
19    17
20    52
dtype: int64

[5] Doublons
2

[6] Aperçu


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,Data extracted on 09/04/2026 14:04:03 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,Long-term residents by citizenship on 31 Decem...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,23/03/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Constats
- Le fichier contient une feuille `Summary` et plusieurs feuilles de données.
- Les vraies données ne commencent pas ? la première ligne.
- Le jeu de données couvre surtout 2020 ? 2024.
- Il sert ? Q3.
- Il permet d'observer les changements de statut migratoire.
- Il faudra commencer avec la feuille la plus simple, probablement ?ge total + sexe total.

## Audit - Eurostat acquisition de citoyennet?
Objectif : vérifier le jeu de données des acquisitions de citoyennet? pour les Camerounais en Europe.

In [26]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx")
xls.sheet_names

/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Summary',
 'Sheet 1',
 'Sheet 2',
 'Sheet 3',
 'Sheet 4',
 'Sheet 5',
 'Sheet 6',
 'Sheet 7',
 'Sheet 8',
 'Sheet 9',
 'Sheet 10',
 'Sheet 11',
 'Sheet 12',
 'Sheet 13',
 'Sheet 14',
 'Sheet 15',
 'Sheet 16',
 'Sheet 17',
 'Sheet 18',
 'Sheet 19',
 'Sheet 20',
 'Sheet 21',
 'Sheet 22',
 'Sheet 23',
 'Sheet 24',
 'Sheet 25',
 'Sheet 26',
 'Sheet 27',
 'Sheet 28',
 'Sheet 29',
 'Sheet 30',
 'Sheet 31',
 'Sheet 32',
 'Sheet 33',
 'Sheet 34',
 'Sheet 35',
 'Sheet 36',
 'Sheet 37',
 'Sheet 38',
 'Sheet 39',
 'Sheet 40',
 'Sheet 41',
 'Sheet 42',
 'Sheet 43',
 'Sheet 44',
 'Sheet 45',
 'Sheet 46',
 'Sheet 47',
 'Sheet 48',
 'Sheet 49',
 'Sheet 50',
 'Sheet 51',
 'Sheet 52',
 'Sheet 53',
 'Sheet 54',
 'Sheet 55',
 'Sheet 56',
 'Sheet 57',
 'Sheet 58',
 'Sheet 59',
 'Sheet 60',
 'Sheet 61',
 'Sheet 62',
 'Sheet 63',
 'Sheet 64',
 'Sheet 65',
 'Sheet 66',
 'Sheet 67',
 'Sheet 68',
 'Sheet 69',
 'Sheet 70',
 'Sheet 71',
 'Sheet 72',
 'Sheet 73',
 'Sheet 74',
 'Sheet 75',
 'Sheet 76',
 'Sheet 7

In [27]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw.head(15)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Data extracted on 09/04/2026 14:08:47 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,"Acquisition of citizenship by age group, sex a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,27/03/2026 11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Country of citizenship,NaN,Cameroon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Age definition,NaN,Age reached during the year,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Age class,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Unit of measure,NaN,Number,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Sex,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
audit_dataset(
    df=sheet1_raw,
    dataset_name="Eurostat citizenship acquisition - Sheet 1 (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : Eurostat citizenship acquisition - Sheet 1 (raw)

[1] Taille du dataset
Lignes : 64
Colonnes : 20

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

[3] Types
0         str
1      object
2         str
3      object
4         str
5      object
6         str
7      object
8         str
9      object
10        str
11     object
12        str
13     object
14        str
15     object
16    float64
17     object
18    float64
19     object
dtype: object

[4] Valeurs manquantes
0      3
1     13
2     56
3     20
4     62
5     20
6     60
7     20
8     59
9     20
10    60
11    20
12    62
13    20
14    62
15    20
16    64
17    20
18    64
19    20
dtype: int64

[5] Doublons
2

[6] Aperçu


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Data extracted on 09/04/2026 14:08:47 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,"Acquisition of citizenship by age group, sex a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,27/03/2026 11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Constats
- Le fichier contient une feuille `Summary` et de nombreuses feuilles de données.
- Les vraies données ne commencent pas ? la première ligne.
- Le jeu de données couvre 2015 ? 2024.
- Il sert ? Q3.
- Il permet d'observer la naturalisation / acquisition de citoyennet?.
- Il faudra commencer par la feuille la plus simple : ?ge total + sexe total.

## Audit - StatCan statut d'immigration des Camerounais
Objectif : vérifier le fichier de contexte StatCan sur le statut d'immigration des Camerounais au Canada.

In [29]:
CANADA_PATH = DATA_RAW / "canada"

In [31]:
statcan_raw = pd.read_csv(
    CANADA_PATH / "statcan_cameroon_immigrant_status.csv",
    header=None,
    skiprows=8
)

statcan_raw.head(15)

,0,1,2,3,4,5,6,7,8,9,10,11
0,Geography,Canada i1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Citizenship (9) 1,Total - Citizenship 6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Age (8D) 2,Total - Age,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gender (3) 3 4,Total - Gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Statistics (3),Count,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Immigrant status and period of immigration (11),Total - Immigrant status and period of immigra...,Non-immigrants 8,Immigrants 9,Before 1980,1980 to 1990,1991 to 2000,2001 to 2010,2011 to 2021 10,2011 to 2015,2016 to 2021 11,Non-permanent residents 12
6,Place of birth (290) 5,2021,2021,2021,2021,2021,2021,2021,2021,2021,2021,2021
7,Total – Place of birth 13,"36,328,475","27,042,125","8,361,505","1,540,615","924,320","1,511,480","1,930,525","2,454,570","1,126,330","1,328,240","924,850"
8,Newfoundland and Labrador,"636,150","636,090",60,35,20,0,0,0,0,0,0
9,Prince Edward Island,"140,505","140,495",15,10,0,0,0,0,0,0,0


In [32]:
audit_dataset(
    df=statcan_raw,
    dataset_name="StatCan - raw immigrant status",
    year_col=None,
    country_cols=None
)

AUDIT : StatCan - raw immigrant status

[1] Taille du dataset
Lignes : 336
Colonnes : 12

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

[3] Types
0     str
1     str
2     str
3     str
4     str
5     str
6     str
7     str
8     str
9     str
10    str
11    str
dtype: object

[4] Valeurs manquantes
0      0
1      7
2     45
3     45
4     45
5     45
6     45
7     45
8     45
9     45
10    45
11    45
dtype: int64

[5] Doublons
0

[6] Aperçu


,0,1,2,3,4,5,6,7,8,9,10,11
0,Geography,Canada i1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Citizenship (9) 1,Total - Citizenship 6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Age (8D) 2,Total - Age,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gender (3) 3 4,Total - Gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Statistics (3),Count,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Conclusion d'audit
- Jeu de données utile comme contexte Canada.
- Non prioritaire pour l'?volution annuelle.
- Nettoyage nécessaire ? cause des métadonnées en tête de fichier.

## Audit - IRCC résidents permanents
Objectif : vérifier le jeu de données des résidents permanents camerounais au Canada.

In [33]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_permanent_residents_cameroon.xlsx")
xls.sheet_names

['PR - CITZ']

In [34]:
ircc_pr_raw = pd.read_excel(
    CANADA_PATH / "ircc_permanent_residents_cameroon.xlsx",
    sheet_name=0,
    header=None
)

ircc_pr_raw.head(15)

,0,1,2,3,4,5,6,7,8,9,...,181,182,183,184,185,186,187,188,189,190
0,Canada - Admissions of Permanent Residents by ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country of Citizenship,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,Q1,NaN,NaN,NaN,Q2,NaN,NaN,NaN,Q3,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,Jun,Q2 Total,Jul,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN
5,Afghanistan,90,125,220,435,135,220,195,555,250,...,720,"2,470",945,880,735,"2,560","8,215",230,230,230
6,Albania,15,25,50,90,45,45,75,165,45,...,40,150,45,25,20,90,485,20,20,20
7,Algeria,80,125,235,440,255,340,280,875,310,...,820,"2,425",590,380,420,"1,390","7,600",365,365,365
8,Andorra,0,0,0,0,--,0,0,--,0,...,0,0,0,0,0,0,--,--,--,--
9,Angola,--,0,--,--,10,--,--,15,--,...,15,80,20,10,25,55,205,15,15,15


In [35]:
audit_dataset(
    df=ircc_pr_raw,
    dataset_name="IRCC - Permanent residents (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : IRCC - Permanent residents (raw)

[1] Taille du dataset
Lignes : 228
Colonnes : 191

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190]

[3] Types
0      str
1      str
2      str
3   

,0,1,2,3,4,5,6,7,8,9,...,181,182,183,184,185,186,187,188,189,190
0,Canada - Admissions of Permanent Residents by ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country of Citizenship,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,Q1,NaN,NaN,NaN,Q2,NaN,NaN,NaN,Q3,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,Jun,Q2 Total,Jul,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN


In [36]:
ircc_pr_raw[ircc_pr_raw.astype(str).apply(lambda col: col.str.contains("Cameroon", case=False, na=False)).any(axis=1)]

,0,1,2,3,4,5,6,7,8,9,...,181,182,183,184,185,186,187,188,189,190
36,"Cameroon, Federal Republic of",75,85,150,310,150,160,200,510,315,...,"2,585","6,005","2,650","1,975","1,940","6,560","22,925","1,150","1,150","1,150"


### Constats
- Fichier Excel avec en-tête multi-lignes.
- Structure large.
- Couverture 2015 ? 2026.
- Présence de la ligne `Cameroon, Federal Republic of`.
- Présence possible de valeurs `--`.
- Jeu de données principal pour le Canada.

## Audit - IRCC résidents permanents par catégorie
Objectif : vérifier le jeu de données des résidents permanents camerounais par catégorie d'immigration.

In [37]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_permanent_residents_cameroon_by_category.xlsx")
xls.sheet_names

['PR - ImmCat']

In [38]:
ircc_pr_cat_raw = pd.read_excel(
    CANADA_PATH / "ircc_permanent_residents_cameroon_by_category.xlsx",
    sheet_name=0,
    header=None
)

ircc_pr_cat_raw.head(20)

,0,1,2,3,4,5,6,7,8,9,...,184,185,186,187,188,189,190,191,192,193
0,Canada - Admissions of Permanent Residents by ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country of Citizenship and Immigration Category,NaN,NaN,NaN,2015,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,NaN,NaN,NaN,Q1,NaN,NaN,NaN,Q2,NaN,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,NaN,NaN,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN
5,NaN,NaN,NaN,Atlantic Immigration Pilot Programs,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,NaN,NaN,NaN,Canadian Experience,0,0,0,0,0,0,...,0,0,0,0,0,0,--,0,0,0
7,NaN,NaN,NaN,Caregiver,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,NaN,NaN,NaN,Federal Economic Mobility Pathways Pilot,0,0,0,0,0,0,...,10,10,--,5,--,10,30,0,0,0
9,NaN,NaN,NaN,Skilled Worker,5,--,0,5,0,0,...,0,0,0,0,0,0,--,0,0,0


In [39]:
audit_dataset(
    df=ircc_pr_cat_raw,
    dataset_name="IRCC - Permanent residents by category (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : IRCC - Permanent residents by category (raw)

[1] Taille du dataset
Lignes : 5345
Colonnes : 194

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193]

[3] Types
0      st

,0,1,2,3,4,5,6,7,8,9,...,184,185,186,187,188,189,190,191,192,193
0,Canada - Admissions of Permanent Residents by ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country of Citizenship and Immigration Category,NaN,NaN,NaN,2015,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,NaN,NaN,NaN,Q1,NaN,NaN,NaN,Q2,NaN,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,NaN,NaN,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN


In [40]:
ircc_pr_cat_raw[
    ircc_pr_cat_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

,0,1,2,3,4,5,6,7,8,9,...,184,185,186,187,188,189,190,191,192,193
803,"Cameroon, Federal Republic of - Total",NaN,NaN,NaN,75,85,150,310,150,160,...,"2,585","6,005","2,650","1,975","1,940","6,560","22,925","1,150","1,150","1,150"
5153,"Cameroon, Federal Republic of",NaN,NaN,NaN,75,85,150,310,150,160,...,"2,585","6,005","2,650","1,975","1,940","6,560","22,925","1,150","1,150","1,150"


### Constats
- Fichier Excel avec en-tête multi-lignes.
- Structure large.
- Couverture 2015 ? 2026.
- Présence d'un bloc `Cameroon, Federal Republic of - Total`.
- Présence de sous-catégories d'immigration sous le total.
- Présence possible de valeurs `--`.
- Jeu de données très utile pour enrichir Q2 et Q3 côté Canada.

### Conclusion d'audit
- Jeu de données principal pour le Canada.
- Très utile pour les catégories d'admission.
- Nettoyage nécessaire : reconstruire l'en-tête, extraire le bloc Cameroun, gérer les catégories et passer en format long.

## Audit - IRCC permis d'?tudes
Objectif : vérifier le jeu de données des permis d'?tudes des Camerounais au Canada.

In [41]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_study_permits_cameroon.xlsx")
xls.sheet_names

['TR - SP CITZ']

In [42]:
ircc_study_raw = pd.read_excel(
    CANADA_PATH / "ircc_study_permits_cameroon.xlsx",
    sheet_name=0,
    header=None
)

ircc_study_raw.head(20)

,0,1,2,3,4,5,6,7,8,9,...,181,182,183,184,185,186,187,188,189,190
0,Canada – Study permit holders by country of ci...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country of Citizenship,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,Q1,NaN,NaN,NaN,Q2,NaN,NaN,NaN,Q3,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,Jun,Q2 Total,Jul,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN
5,Afghanistan,10,--,--,15,10,5,5,20,10,...,20,60,15,15,10,40,135,35,35,35
6,Albania,10,5,--,15,10,5,5,25,10,...,45,155,30,35,40,105,420,25,25,25
7,Algeria,60,40,55,155,55,55,45,155,120,...,"1,005","2,615",615,920,765,"2,280","7,705",400,400,400
8,Andorra,--,0,0,--,0,0,--,--,--,...,--,--,0,0,0,0,5,0,0,0
9,Angola,15,--,--,15,0,--,--,5,--,...,20,45,10,10,5,30,100,10,10,10


In [43]:
audit_dataset(
    df=ircc_study_raw,
    dataset_name="IRCC - Study permits (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : IRCC - Study permits (raw)

[1] Taille du dataset
Lignes : 233
Colonnes : 191

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190]

[3] Types
0      str
1      str
2      str
3      str

,0,1,2,3,4,5,6,7,8,9,...,181,182,183,184,185,186,187,188,189,190
0,Canada – Study permit holders by country of ci...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Country of Citizenship,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,Q1,NaN,NaN,NaN,Q2,NaN,NaN,NaN,Q3,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,Jun,Q2 Total,Jul,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN


In [44]:
ircc_study_raw[
    ircc_study_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

,0,1,2,3,4,5,6,7,8,9,...,181,182,183,184,185,186,187,188,189,190
39,"Cameroon, Federal Republic of",95,50,55,200,55,65,60,180,115,...,580,"1,665",375,420,370,"1,160","4,590",300,300,300


### Constats
- Fichier Excel avec en-tête multi-lignes.
- Structure large.
- Couverture 2015 ? 2026.
- Présence de la ligne `Cameroon, Federal Republic of`.
- Présence possible de valeurs `--`.
- Jeu de données principal pour Q2 côté Canada.

### Conclusion d'audit
- Jeu de données principal pour le Canada.
- Très utile pour analyser les permis d'?tudes.
- Nettoyage nécessaire : reconstruire l'en-tête, passer en format long, gérer `--`.

## Audit - IRCC temporaire vers permanent
Objectif : vérifier le jeu de données des transitions du statut temporaire vers la résidence permanente au Canada.

In [45]:
xls = pd.ExcelFile(CANADA_PATH / "ircc_temp_to_pr_cameroon.xlsx")
xls.sheet_names

['TR to PR - IS']

In [46]:
ircc_temp_pr_raw = pd.read_excel(
    CANADA_PATH / "ircc_temp_to_pr_cameroon.xlsx",
    sheet_name=0,
    header=None
)

ircc_temp_pr_raw.head(20)

,0,1,2,3,4,5,6,7,8,9,...,184,185,186,187,188,189,190,191,192,193
0,Canada – Admissions of Permanent Residents wit...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Province/Territory and Immigration Category,NaN,NaN,NaN,2015,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,NaN,NaN,NaN,Q1,NaN,NaN,NaN,Q2,NaN,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,NaN,NaN,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN
5,NaN,NaN,NaN,Atlantic Immigration Pilot Programs,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,NaN,NaN,NaN,Atlantic Immigration Programs,0,0,0,0,0,0,...,20,45,15,20,15,45,150,--,--,--
7,NaN,NaN,NaN,Canadian Experience,0,0,0,0,0,0,...,0,--,0,0,0,0,5,--,--,--
8,NaN,NaN,NaN,Caregiver,0,0,0,0,0,0,...,0,0,0,0,0,0,--,0,0,0
9,NaN,NaN,NaN,Skilled Trade,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
audit_dataset(
    df=ircc_temp_pr_raw,
    dataset_name="IRCC - Temporary to permanent (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : IRCC - Temporary to permanent (raw)

[1] Taille du dataset
Lignes : 299
Colonnes : 194

[2] Colonnes
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193]

[3] Types
0      str
1      s

,0,1,2,3,4,5,6,7,8,9,...,184,185,186,187,188,189,190,191,192,193
0,Canada – Admissions of Permanent Residents wit...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Province/Territory and Immigration Category,NaN,NaN,NaN,2015,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,NaN,NaN
3,NaN,NaN,NaN,NaN,Q1,NaN,NaN,NaN,Q2,NaN,...,NaN,NaN,Q4,NaN,NaN,NaN,2025 Total,Q1,NaN,2026 Total
4,NaN,NaN,NaN,NaN,Jan,Feb,Mar,Q1 Total,Apr,May,...,Sep,Q3 Total,Oct,Nov,Dec,Q4 Total,NaN,Jan,Q1 Total,NaN


In [48]:
ircc_temp_pr_raw[
    ircc_temp_pr_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

,0,1,2,3,4,5,6,7,8,9,...,184,185,186,187,188,189,190,191,192,193


### Constats
- Fichier Excel avec en-tête multi-lignes.
- Structure large.
- Couverture 2015 ? 2026.
- Le fichier semble organis? par province/territoire et catégorie.
- Vérifier si `Cameroon` est réellement présent ou non.
- Si `Cameroon` n'est pas présent, ce fichier sera plutôt un contexte Canada qu'un jeu de données Cameroun direct.

### Conclusion d'audit
- Jeu de données intéressant pour comprendre la logique de transition du temporaire vers le permanent au Canada.
- ? garder si utile comme contexte.
- Si `Cameroon` est absent, ne pas le considérer comme jeu de données principal Cameroun.

## Audit - USA résidents permanents légaux
Objectif : vérifier le jeu de données DHS sur les résidents permanents liés au Cameroun.

In [49]:
USA_PATH = DATA_RAW / "usa"

In [50]:
xls = pd.ExcelFile(USA_PATH / "usa_lawful_permanent_residents_cameroon.xlsx")
xls.sheet_names

['TOC',
 'Table 1',
 'Table 2',
 'Table 3d',
 'Table 4',
 'Table 5',
 'Table 6',
 'Table 7d',
 'Table 8',
 'Table 9d',
 'Table 10d',
 'Table 11d',
 'Table 12d',
 'LPRSuppTable 1d',
 'LPRSuppTable 2d',
 'LPRSuppTable 3d',
 'LPRSuppTable 4d']

In [51]:
usa_lpr_raw = pd.read_excel(
    USA_PATH / "usa_lawful_permanent_residents_cameroon.xlsx",
    sheet_name=0,
    header=None
)

usa_lpr_raw.head(20)

,0,1
0,Table of Contents,NaN
1,NaN,NaN
2,Click link for corresponding tab:,NaN
3,Table 1,Persons Obtaining Lawful Permanent Resident St...
4,Table 2,Persons Obtaining Lawful Permanent Resident St...
5,Table 3d,Persons Obtaining Lawful Permanent Resident St...
6,Table 4,Persons Obtaining Lawful Permanent Resident St...
7,Table 5,Persons Obtaining Lawful Permanent Resident St...
8,Table 6,Persons Obtaining Lawful Permanent Resident St...
9,Table 7d,Persons Obtaining Lawful Permanent Resident St...


In [52]:
audit_dataset(
    df=usa_lpr_raw,
    dataset_name="USA DHS - Lawful permanent residents (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : USA DHS - Lawful permanent residents (raw)

[1] Taille du dataset
Lignes : 19
Colonnes : 2

[2] Colonnes
[0, 1]

[3] Types
0    str
1    str
dtype: object

[4] Valeurs manquantes
0    1
1    3
dtype: int64

[5] Doublons
0

[6] Aperçu


,0,1
0,Table of Contents,NaN
1,NaN,NaN
2,Click link for corresponding tab:,NaN
3,Table 1,Persons Obtaining Lawful Permanent Resident St...
4,Table 2,Persons Obtaining Lawful Permanent Resident St...


In [53]:
usa_lpr_raw[
    usa_lpr_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

,0,1


### Constats
- Fichier Excel avec plusieurs tables / feuilles.
- Les données DHS sont souvent organisées par table.
- Présence de lignes `Cameroon` ? vérifier selon les feuilles.
- Couverture utile jusqu'en 2022.
- Attention : les années DHS sont des années fiscales.
- Jeu de données utile pour Q3 et pour le zoom USA.

### Conclusion d'audit
- Jeu de données utile pour les ?tats-Unis.
- Il ne couvre pas toute la période 2015-2024 jusqu'au bout.
- Attention aux années fiscales.
- Nettoyage nécessaire : identifier la bonne table, extraire la ligne `Cameroon`, harmoniser les années.

## Audit - USA arrivées de réfugiés
Objectif : vérifier le jeu de données DHS / OHSS sur les arrivées de réfugiés camerounais aux ?tats-Unis.

In [54]:
xls = pd.ExcelFile(USA_PATH / "usa_refugee_arrivals_cameroon.xlsx")
xls.sheet_names

['TOC', 'Table 13', 'Table 14', 'Table 15']

In [55]:
usa_refugees_raw = pd.read_excel(
    USA_PATH / "usa_refugee_arrivals_cameroon.xlsx",
    sheet_name=0,
    header=None
)

usa_refugees_raw.head(20)

,0,1
0,Table of Contents,NaN
1,NaN,NaN
2,Click link for corresponding tab:,NaN
3,Table 13,Refugee Arrivals: Fiscal Years 1980 to 2024
4,Table 14,Refugee Arrivals by Region and Country of Nati...
5,Table 15,Refugee Arrivals by Relationship to Principal ...


In [56]:
audit_dataset(
    df=usa_refugees_raw,
    dataset_name="USA DHS - Refugee arrivals (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : USA DHS - Refugee arrivals (raw)

[1] Taille du dataset
Lignes : 6
Colonnes : 2

[2] Colonnes
[0, 1]

[3] Types
0    str
1    str
dtype: object

[4] Valeurs manquantes
0    1
1    3
dtype: int64

[5] Doublons
0

[6] Aperçu


,0,1
0,Table of Contents,NaN
1,NaN,NaN
2,Click link for corresponding tab:,NaN
3,Table 13,Refugee Arrivals: Fiscal Years 1980 to 2024
4,Table 14,Refugee Arrivals by Region and Country of Nati...


In [57]:
usa_refugees_raw[
    usa_refugees_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

,0,1


### Constats
- Fichier Excel DHS / OHSS.
- Présence de tables par année ou par série.
- La ligne `Cameroon` doit ?tre repérée dans la table utile.
- Couverture 2015 ? 2024.
- Attention : les années DHS sont des années fiscales.
- Jeu de données utile pour l'angle réfugiés / asile côté USA.

### Conclusion d'audit
- Jeu de données utile pour complèter le bloc USA.
- Angle spécifique : réfugiés.
- Attention ? ne pas confondre ce jeu de données avec toute la migration camerounaise vers les USA.
- Nettoyage nécessaire : identifier la bonne table, extraire `Cameroon`, harmoniser les années fiscales.

## Audit - USA ACS population sélectionnée
Objectif : vérifier le fichier ACS de profil descriptif des Camerounais / de la population liée au Cameroun aux ?tats-Unis.

In [58]:
xls = pd.ExcelFile(USA_PATH / "usa_acs_selected_population_cameroon.xlsx")
xls.sheet_names

['Information', 'Data']

In [59]:
usa_acs_raw = pd.read_excel(
    USA_PATH / "usa_acs_selected_population_cameroon.xlsx",
    sheet_name=0,
    header=None
)

usa_acs_raw.head(20)

,0,1
0,Selected Population Profile in the United States,NaN
1,NaN,NaN
2,Note: The table shown may have been modified b...,NaN
3,NaN,NaN
4,DATA NOTES,NaN
5,TABLE ID:,S0201
6,SURVEY/PROGRAM:,American Community Survey
7,VINTAGE:,2024
8,DATASET:,ACSSPP1Y2024
9,PRODUCT:,ACS 1-Year Estimates Selected Population Profiles


In [60]:
audit_dataset(
    df=usa_acs_raw,
    dataset_name="USA ACS - Selected population (raw)",
    year_col=None,
    country_cols=None
)

AUDIT : USA ACS - Selected population (raw)

[1] Taille du dataset
Lignes : 60
Colonnes : 2

[2] Colonnes
[0, 1]

[3] Types
0    str
1    str
dtype: object

[4] Valeurs manquantes
0    12
1    31
dtype: int64

[5] Doublons
11

[6] Aperçu


,0,1
0,Selected Population Profile in the United States,NaN
1,NaN,NaN
2,Note: The table shown may have been modified b...,NaN
3,NaN,NaN
4,DATA NOTES,NaN


In [61]:
usa_acs_raw[
    usa_acs_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
]

,0,1
16,TOPICS,Cameroon


### Constats
- Fichier ACS / Census.
- Probablement une feuille de données avec un profil descriptif.
- Point de contexte, pas une vraie série annuelle longue.
- Utile pour enrichir le profil USA.
- Présence possible de marges d'erreur.
- Jeu de données secondaire / de contexte plutôt que central.

### Conclusion d'audit
- Jeu de données utile comme contexte descriptif USA.
- Pas un jeu de données principal pour l'?volution 2015-2024.
- ? garder en complément.
- Nettoyage nécessaire : identifier la vraie table utile, conserver l'estimation et la marge d'erreur si pertinentes.

## Audit - visas Japon Cameroun
Objectif : vérifier le fichier des visas liés aux Camerounais au Japon.

In [62]:
JAPAN_PATH = DATA_RAW / "japan"

In [63]:
japan_visas_raw = pd.read_csv(
    JAPAN_PATH / "japan_visas_cameroon.csv"
)

japan_visas_raw.head(20)

,Year,Regional code,Country,Number of issued,Number of issued_numerical,Travel certificate,Diplomacy,Public use,Passing,Short -term stay,...,identification,Specific_housework employee,Specified_short term,Specific_profit representative staff,Specific_working holiday,Specified_Amasport,"Specific_Japanese spouse, etc.","Specific_Permanent resident's spouse, etc.",Specified_Distingant,Specific_Others
0,2017,0,total,5869012,741415,389.0,4862.0,16402.0,8518,5392838,...,53552,122.0,NaN,49.0,15521.0,77.0,10227,2181.0,16945,8431.0
1,2017,40,Afghanistan,1005,46,0.0,14.0,112.0,0,224,...,13,0.0,NaN,0.0,0.0,0.0,2,3.0,8,0.0
2,2017,80,Albania,252,7,0.0,0.0,2.0,1,200,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0
3,2017,120,Algeria,899,19,0.0,8.0,11.0,3,813,...,2,0.0,NaN,0.0,0.0,0.0,1,1.0,0,0.0
4,2017,200,Andra,5,0,0.0,0.0,0.0,0,0,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0
5,2017,240,Angola,111,12,75.0,9.0,7.0,0,67,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0
6,2017,280,Antigua Berbuda,41,8,0.0,1.0,0.0,0,28,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0
7,2017,310,Azerbaijan,619,8,0.0,31.0,20.0,0,505,...,2,0.0,NaN,0.0,0.0,0.0,1,1.0,0,0.0
8,2017,320,Argentina,450,0,0.0,0.0,1.0,0,0,...,93,0.0,NaN,0.0,45.0,0.0,19,2.0,25,2.0
9,2017,360,Australia,3887,281,0.0,54.0,220.0,0,5,...,1341,0.0,NaN,0.0,1182.0,21.0,114,4.0,2,18.0


In [64]:
audit_dataset(
    df=japan_visas_raw,
    dataset_name="Japan visas Cameroon",
    year_col=None,
    country_cols=None
)

AUDIT : Japan visas Cameroon

[1] Taille du dataset
Lignes : 1945
Colonnes : 48

[2] Colonnes
['Year', 'Regional code', 'Country', 'Number of issued', 'Number of issued_numerical', 'Travel certificate', 'Diplomacy', 'Public use', 'Passing', 'Short -term stay', 'Staying in medical care', 'Advanced profession', 'Employment', 'Employment_Professor', 'Employment_Art', 'Employment_religion', 'Employment_Report', 'Employment_Management / Management', 'Employment_Law accounting', 'Employment_Medical care', 'Employment_Research', 'Employment_Education', 'Employment_Country', 'Employment_Technology', 'Employment_Humanities International', 'Employment_C corporate rotation', 'Employment_show', 'Employment_Skills', 'Employment_Steng (new)', 'Employment_Nursing care', 'Ordinary', 'General_Cultural activity', 'General_Studying abroad', 'General_Arademy', 'General_Training', 'General_Family stay', 'General_Stifying 1 I', 'General_Steng 1', 'identification', 'Specific_housework employee', 'Specified_s

,Year,Regional code,Country,Number of issued,Number of issued_numerical,Travel certificate,Diplomacy,Public use,Passing,Short -term stay,...,identification,Specific_housework employee,Specified_short term,Specific_profit representative staff,Specific_working holiday,Specified_Amasport,"Specific_Japanese spouse, etc.","Specific_Permanent resident's spouse, etc.",Specified_Distingant,Specific_Others
0,2017,0,total,5869012,741415,389.0,4862.0,16402.0,8518,5392838,...,53552,122.0,NaN,49.0,15521.0,77.0,10227,2181.0,16945,8431.0
1,2017,40,Afghanistan,1005,46,0.0,14.0,112.0,0,224,...,13,0.0,NaN,0.0,0.0,0.0,2,3.0,8,0.0
2,2017,80,Albania,252,7,0.0,0.0,2.0,1,200,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0
3,2017,120,Algeria,899,19,0.0,8.0,11.0,3,813,...,2,0.0,NaN,0.0,0.0,0.0,1,1.0,0,0.0
4,2017,200,Andra,5,0,0.0,0.0,0.0,0,0,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0


In [65]:
japan_visas_raw.columns.tolist()

['Year',
 'Regional code',
 'Country',
 'Number of issued',
 'Number of issued_numerical',
 'Travel certificate',
 'Diplomacy',
 'Public use',
 'Passing',
 'Short -term stay',
 'Staying in medical care',
 'Advanced profession',
 'Employment',
 'Employment_Professor',
 'Employment_Art',
 'Employment_religion',
 'Employment_Report',
 'Employment_Management / Management',
 'Employment_Law accounting',
 'Employment_Medical care',
 'Employment_Research',
 'Employment_Education',
 'Employment_Country',
 'Employment_Technology',
 'Employment_Humanities International',
 'Employment_C corporate rotation',
 'Employment_show',
 'Employment_Skills',
 'Employment_Steng (new)',
 'Employment_Nursing care',
 'Ordinary',
 'General_Cultural activity',
 'General_Studying abroad',
 'General_Arademy',
 'General_Training',
 'General_Family stay',
 'General_Stifying 1 I',
 'General_Steng 1',
 'identification',
 'Specific_housework employee',
 'Specified_short term',
 'Specific_profit representative staff',
 

In [66]:
japan_visas_raw[
    japan_visas_raw.astype(str)
    .apply(lambda col: col.str.contains("Cameroon", case=False, na=False))
    .any(axis=1)
].head(20)

,Year,Regional code,Country,Number of issued,Number of issued_numerical,Travel certificate,Diplomacy,Public use,Passing,Short -term stay,...,identification,Specific_housework employee,Specified_short term,Specific_profit representative staff,Specific_working holiday,Specified_Amasport,"Specific_Japanese spouse, etc.","Specific_Permanent resident's spouse, etc.",Specified_Distingant,Specific_Others
30,2017,1200,Cameroon,583,26,0.0,8.0,68.0,2,353,...,1,0.0,NaN,0.0,0.0,0.0,0,1.0,0,0.0
232,2016,1200,Cameroon,362,12,0.0,11.0,54.0,1,203,...,5,0.0,NaN,0.0,0.0,0.0,3,1.0,0,1.0
434,2015,1200,Cameroon,412,16,0.0,5.0,57.0,1,273,...,5,0.0,NaN,0.0,0.0,0.0,1,2.0,1,1.0
637,2014,1200,Cameroon,308,17,0.0,9.0,56.0,4,177,...,1,0.0,NaN,0.0,0.0,0.0,0,1.0,0,0.0
844,2013,1200,Cameroon,491,107,26.0,35.0,120.0,5,270,...,2,0.0,NaN,0.0,0.0,0.0,1,1.0,0,0.0
1052,2012,1200,Cameroon,435,7,0.0,29.0,47.0,1,283,...,4,0.0,NaN,0.0,0.0,0.0,3,0.0,0,1.0
1260,2011,1200,Cameroon,368,13,0.0,8.0,14.0,5,283,...,0,0.0,NaN,0.0,0.0,0.0,0,0.0,0,0.0
1468,2010,1200,Cameroon,414,34,0.0,13.0,26.0,4,286,...,2,0.0,NaN,0.0,0.0,0.0,2,0.0,0,0.0
1675,2009,1200,Cameroon,421,16,0.0,7.0,4.0,33,296,...,3,0.0,NaN,0.0,0.0,0.0,1,0.0,0,2.0


### Constats
- Fichier CSV plus simple que les fichiers Excel Canada/USA.
- Jeu de données utile pour un complément Asie.
- Probablement plus limit? en période que les blocs Europe/Canada.
- Utile surtout comme ouverture internationale, pas comme cœur analytique principal.